In [ ]:
import os
import shutil

# Fix 1: Removed the double-quote wrapper around paths
in_folder = "D:/BMEN_600/ADNI/MCI/"
output_folder = "D:/BMEN_600/hippunfold/input/MCI/"

# Fix 2: Use os.makedirs with exist_ok=True so it doesn't crash if folders already exist
f1 = os.path.join(output_folder, "1")
os.makedirs(f1, exist_ok=True)

# Loop through each subject folder (e.g., sub-001, sub-002) in the AD folder
for name in os.listdir(in_folder):
    subject_source_dir = os.path.join(in_folder, name)
    
    # Ensure we are only looking at directories, skipping random files
    if os.path.isdir(subject_source_dir):
        
        # Build the exact deep nested path to find the target file
        # Matches: in_folder/sub-001/1/2/anat/sub-001_acq-mprage_T1w.nii.gz
        source_file = os.path.join(subject_source_dir, "1", "2", "anat", f"{name}_acq-mprage_T1w.nii.gz")
        
        # Build the target destination path inside hippunfold
        # Matches: output_folder/1/sub-001/anat/
        target_anat_dir = os.path.join(f1, name, "anat")
        os.makedirs(target_anat_dir, exist_ok=True)
        
        target_file = os.path.join(target_anat_dir, f"{name}_acq-mprage_T1w.nii.gz")
        
        # Check if the source file actually exists before trying to move it
        if os.path.exists(source_file):
            try:
                # Use shutil.move to safely transfer the structural scans
                shutil.move(source_file, target_file)
                print(f"Successfully structured and moved: {name}")
            except Exception as e:
                print(f"Error moving file for {name}: {e}")
        else:
            print(f"Warning: Expected file not found at {source_file}")

print("\nDataset resturcturing complete!")



In [ ]:
import os
import glob
import shutil
import dicom2nifti

# Set the top-level directory where all your subject folders live
base_directory = "D:/BMEN_600/ADNI/MCI"

# Iterate through the nested directory structure
for root, dirs, files in os.walk(base_directory):
    # Check if the current folder is named 'anat'
    if os.path.basename(root) == "anat":
        
        # Parse the subject name from the path
        rel_path = os.path.relpath(root, base_directory)
        path_parts = rel_path.split(os.sep)
        subject_name = path_parts[0]  # The top-level folder under base_directory
        
        # Create a temporary workspace folder inside 'anat' to avoid scanning issues
        temp_out_dir = os.path.join(root, "temp_nifti_out")
        os.makedirs(temp_out_dir, exist_ok=True)
        
        print(f"Processing Subject: {subject_name}")
        print(f"Converting DICOMs in: {root}")
        
        try:
            # 1. Convert DICOMs into the temporary subfolder
            dicom2nifti.convert_directory(
                root, 
                temp_out_dir, 
                compression=True, 
                reorient=True
            )
            
            # 2. Find the generated file ending in .nii.gz
            generated_files = glob.glob(os.path.join(temp_out_dir, "*.nii.gz"))
            
            if generated_files:
                temp_output_file = generated_files[0]
                
                # Define the final path inside the 'anat' folder (e.g., .../anat/Subject_01_anat.nii.gz)
                final_output_file = os.path.join(root, f"{subject_name}_acq-mprage_T1w.nii.gz")
                
                # Move and rename the file out of the temp folder into 'anat'
                shutil.move(temp_output_file, final_output_file)
                print(f"Successfully saved to: {final_output_file}\n")
            else:
                print("Conversion finished, but no NIfTI file was generated.\n")
                
        except Exception as e:
            print(f"Failed to convert {root}. Error: {e}\n")
            
        finally:
            # Clean up the temporary directory
            if os.path.exists(temp_out_dir):
                shutil.rmtree(temp_out_dir)

print("All subjects processed!")

In [ ]:
from pathlib import Path

# 1. Define your exact BIDS root directory 
root_dir = Path("C:/Users/adith/hippunfold/input/MCI")

print("Starting file renaming process...\n")

# 2. Search recursively through all subdirectories for 'sub-*/anat'
for anat_folder in root_dir.rglob("sub-*/anat"):
    
    # Extract the subject ID dynamically (e.g., "sub-001")
    sub_id = anat_folder.parent.name  
    
    # 3. Define the old and new filenames using your pattern
    old_filename = f"{sub_id}_acq-mprage_T1w.nii.gz"
    new_filename = f"{sub_id}_T1w.nii.gz"
    
    # Get the absolute paths for both files
    old_file_path = anat_folder / old_filename
    new_file_path = anat_folder / new_filename
    
    # 4. Safely check if the file exists and rename it
    if old_file_path.exists():
        old_file_path.rename(new_file_path)
        print(f"✅ Renamed: {old_filename}  --->  {new_filename}")
    else:
        # Optional: alert you if a subject doesn't have that specific file
        print(f"⚠️  Skipped (File not found): {old_file_path.relative_to(root_dir)}")

print("\nProcess finished.")